In [1]:
#Final
import os
from dataclasses import dataclass, field
from pathlib import Path
from typing import List, Optional, Dict, Any
import pandas as pd
import torch

from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

# Maximize multi-core CPU performance
torch.set_num_threads(os.cpu_count() or 4)

@dataclass
class IngestionConfig:
    csv_path: str = "data/processed/train.csv"
    text_column: str = "text"
    id_column: str = "id"
    label_column: str = "label"

    # SPEED TWEAK 1: 700 chars reduces chunk count by ~40% while staying well under 256 tokens
    chunk_size: int = 700
    chunk_overlap: int = 70
    separators: List[str] = field(default_factory=lambda: [
        "\n\n", "\n", ". ", "? ", "! ", "; ", " ", ""
    ])

    model_name: str = "sentence-transformers/all-MiniLM-L6-v2"
    device: str = "cuda" if torch.cuda.is_available() else "cpu"

    # SPEED TWEAK 2: 256 internal batch size for PyTorch
    embed_batch_size: int = 256
    
    # SPEED TWEAK 3: 2,000 batch size for Chroma to minimize disk writes
    persist_directory: str = "vectorstore/chroma_db"
    collection_name: str = "train_collection"
    batch_size: int = 2000


class IngestionPipeline:
    def __init__(self, config: Optional[IngestionConfig] = None):
        self.config = config or IngestionConfig()
        self._splitter: Optional[RecursiveCharacterTextSplitter] = None
        self._embeddings: Optional[HuggingFaceEmbeddings] = None
        self._vector_store: Optional[Chroma] = None

    def load_csv(self) -> pd.DataFrame:
        csv_file = Path(self.config.csv_path)
        if not csv_file.exists():
            raise FileNotFoundError(f"[Error] CSV not found at: {csv_file.resolve()}")

        print(f"[1/4] Loading CSV from: {csv_file}")
        df = pd.read_csv(csv_file)
        df = df.dropna(subset=[self.config.text_column])
        print(f"      Loaded {len(df)} records.")
        return df

    def convert_to_documents(self, df: pd.DataFrame) -> List[Document]:
        print(f"[2/4] Chunking text (chunk_size={self.config.chunk_size})...")
        if self._splitter is None:
            self._splitter = RecursiveCharacterTextSplitter(
                chunk_size=self.config.chunk_size,
                chunk_overlap=self.config.chunk_overlap,
                separators=self.config.separators
            )

        documents: List[Document] = []
        for _, row in df.iterrows():
            content = str(row[self.config.text_column]).strip()
            base_id = str(row[self.config.id_column])
            label = str(row[self.config.label_column]) if pd.notnull(row[self.config.label_column]) else ""

            chunks = self._splitter.split_text(content)
            for idx, chunk in enumerate(chunks):
                documents.append(Document(
                    page_content=chunk,
                    metadata={
                        "id": f"{base_id}_{idx}",
                        "original_id": base_id,
                        "label": label,
                        "chunk_index": idx,
                        "total_chunks": len(chunks)
                    }
                ))

        print(f"      Total chunks generated: {len(documents)}")
        return documents

    def get_embeddings(self) -> HuggingFaceEmbeddings:
        if self._embeddings is None:
            print(f"[3/4] Initializing embeddings on device: {self.config.device}...")
            self._embeddings = HuggingFaceEmbeddings(
                model_name=self.config.model_name,
                model_kwargs={"device": self.config.device},
                encode_kwargs={
                    "normalize_embeddings": True,
                    "batch_size": self.config.embed_batch_size  # Fast batching
                }
            )
        return self._embeddings

    def build_vector_store(self, documents: List[Document]) -> Chroma:
        persist_path = Path(self.config.persist_directory)
        persist_path.mkdir(parents=True, exist_ok=True)

        embeddings = self.get_embeddings()
        print(f"[4/4] Ingesting {len(documents)} chunks into Chroma (batch size = {self.config.batch_size})...")

        vector_store = Chroma(
            collection_name=self.config.collection_name,
            embedding_function=embeddings,
            persist_directory=str(persist_path)
        )

        total_docs = len(documents)
        batch_size = self.config.batch_size

        for i in range(0, total_docs, batch_size):
            batch = documents[i : i + batch_size]
            vector_store.add_documents(documents=batch)
            print(f"      -> Ingested {min(i + batch_size, total_docs)} / {total_docs} chunks...")

        print(f"      Ingestion complete! Persisted to '{persist_path}'.")
        self._vector_store = vector_store
        return vector_store

    def run(self) -> Chroma:
        df = self.load_csv()
        documents = self.convert_to_documents(df)
        return self.build_vector_store(documents)



In [2]:
config = IngestionConfig()
pipeline = IngestionPipeline(config=config)
pipeline.run()

[1/4] Loading CSV from: data\processed\train.csv
      Loaded 25894 records.
[2/4] Chunking text (chunk_size=700)...
      Total chunks generated: 42138
[3/4] Initializing embeddings on device: cpu...


c:\Users\Shubhradeep Roy\Downloads\depression-suicide\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2843.82it/s]


[4/4] Ingesting 42138 chunks into Chroma (batch size = 2000)...
      -> Ingested 2000 / 42138 chunks...
      -> Ingested 4000 / 42138 chunks...
      -> Ingested 6000 / 42138 chunks...
      -> Ingested 8000 / 42138 chunks...
      -> Ingested 10000 / 42138 chunks...
      -> Ingested 12000 / 42138 chunks...
      -> Ingested 14000 / 42138 chunks...
      -> Ingested 16000 / 42138 chunks...
      -> Ingested 18000 / 42138 chunks...
      -> Ingested 20000 / 42138 chunks...
      -> Ingested 22000 / 42138 chunks...
      -> Ingested 24000 / 42138 chunks...
      -> Ingested 26000 / 42138 chunks...
      -> Ingested 28000 / 42138 chunks...
      -> Ingested 30000 / 42138 chunks...
      -> Ingested 32000 / 42138 chunks...
      -> Ingested 34000 / 42138 chunks...
      -> Ingested 36000 / 42138 chunks...
      -> Ingested 38000 / 42138 chunks...
      -> Ingested 40000 / 42138 chunks...
      -> Ingested 42000 / 42138 chunks...
      -> Ingested 42138 / 42138 chunks...
      Ingestion 

In [8]:
# Retrieval pipeline--src/retriever.py
import sys
from dataclasses import dataclass
from pathlib import Path
from typing import List, Dict, Any, Optional

from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma


# ===========================================================================
# 1. Configuration Data Class
# ===========================================================================
@dataclass
class RetrieverConfig:
    """Configuration settings for connecting to and querying Chroma DB."""
    persist_directory: str = "vectorstore/chroma_db"
    collection_name: str = "train_collection"
    model_name: str = "sentence-transformers/all-MiniLM-L6-v2"
    device: str = "cpu"
    default_top_k: int = 3
    deduplicate_by_parent: bool = True


# ===========================================================================
# 2. Main Systematic Retriever Class
# ===========================================================================
class SimilarityRetriever:
    """
    Modular retriever to connect to an existing persisted Chroma vector store,
    perform similarity search, MMR diversity search, metadata filtering,
    and parent-document deduplication.
    """

    def __init__(self, config: Optional[RetrieverConfig] = None):
        self.config = config or RetrieverConfig()
        self._embeddings: Optional[HuggingFaceEmbeddings] = None
        self._vector_store: Optional[Chroma] = None
        
        # Connect to Chroma on initialization
        self._connect_vector_store()

    # -----------------------------------------------------------------------
    # Step A: Load Embeddings and Connect to Vector Store
    # -----------------------------------------------------------------------
    def get_embeddings(self) -> HuggingFaceEmbeddings:
        """Initializes and returns the cached Hugging Face embedding function."""
        if self._embeddings is None:
            self._embeddings = HuggingFaceEmbeddings(
                model_name=self.config.model_name,
                model_kwargs={"device": self.config.device},
                encode_kwargs={"normalize_embeddings": True}
            )
        return self._embeddings

    def _connect_vector_store(self) -> Chroma:
        """Connects to the persisted Chroma database index on disk."""
        db_path = Path(self.config.persist_directory)
        if not db_path.exists():
            raise FileNotFoundError(
                f"[Error] Chroma database directory not found at: '{db_path.resolve()}'. "
                f"Please run ingestion first."
            )

        print(f"[*] Connecting to persisted Chroma DB at: '{db_path.resolve()}'")
        embeddings = self.get_embeddings()

        self._vector_store = Chroma(
            collection_name=self.config.collection_name,
            embedding_function=embeddings,
            persist_directory=str(db_path)
        )
        print(f"[✓] Connected successfully to collection: '{self.config.collection_name}'\n")
        return self._vector_store

    # -----------------------------------------------------------------------
    # Step B: Standard Similarity Search (with deduplication)
    # -----------------------------------------------------------------------
    def similarity_search(
        self,
        query: str,
        top_k: Optional[int] = None,
        label_filter: Optional[str] = None,
        deduplicate: Optional[bool] = None
    ) -> List[Dict[str, Any]]:
        """
        Executes similarity search with distance score calculation.
        """
        k = top_k or self.config.default_top_k
        should_dedup = deduplicate if deduplicate is not None else self.config.deduplicate_by_parent
        
        fetch_k = k * 3 if should_dedup else k
        where_filter = {"label": label_filter} if label_filter else None

        raw_results = self._vector_store.similarity_search_with_score(
            query=query,
            k=fetch_k,
            filter=where_filter
        )

        results: List[Dict[str, Any]] = []
        seen_parents = set()

        for doc, score in raw_results:
            parent_id = doc.metadata.get("original_id", "N/A")

            if should_dedup and parent_id in seen_parents:
                continue

            seen_parents.add(parent_id)

            results.append({
                "chunk_id": doc.metadata.get("id", "N/A"),
                #"original_id": parent_id,
                "label": doc.metadata.get("label", "N/A"),
                #"chunk_index": doc.metadata.get("chunk_index", 0) + 1,
                #"total_chunks": doc.metadata.get("total_chunks", 1),
                "distance_score": round(float(score), 4),
                "content": doc.page_content.strip()
            })

            if len(results) >= k:
                break

        return results

    def format_context_for_prompt(self, retrieved_posts: List[Dict[str, Any]]) -> str:
        """
        Formats retrieved posts into a structured, labeled text block
        ready for insertion into the LLM prompt's {context} variable.
        """
        if not retrieved_posts:
            return "No reference historical posts available."

        context_blocks = []
        for rank, post in enumerate(retrieved_posts, start=1):
            label = str(post.get("label", "unknown")).upper()
            content = post.get("content", post.get("text", "")).strip()
            score = post.get("distance_score", post.get("score", None))

            score_str = f" | Distance Score: {score:.4f}" if score is not None else ""

            block = (
                f"--- Reference Post #{rank} [True Severity: {label}{score_str}] ---\n"
                f"\"{content}\""
            )
            context_blocks.append(block)

        return "\n\n".join(context_blocks)

    

In [9]:


retriever = SimilarityRetriever()
results = retriever.similarity_search("Sorry, I quit", top_k=4)

[*] Connecting to persisted Chroma DB at: 'C:\Users\Shubhradeep Roy\Downloads\depression-suicide\depression-suicide\vectorstore\chroma_db'


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 3769.32it/s]


[✓] Connected successfully to collection: 'train_collection'



In [10]:
retriever.format_context_for_prompt(results)

'--- Reference Post #1 [True Severity: SEVERE | Distance Score: 0.6665] ---\n"I am done I quit"\n\n--- Reference Post #2 [True Severity: MODERATE | Distance Score: 0.7634] ---\n". But it\'s ok, I\'ll quit soon."\n\n--- Reference Post #3 [True Severity: MODERATE | Distance Score: 1.0225] ---\n"sorry i exist"\n\n--- Reference Post #4 [True Severity: SEVERE | Distance Score: 1.0742] ---\n". I am not going to quit atleast not yet, I know if this does not work I am getting kicked out of my home (moms words not mine). So yeah it is kind of life and death lol. If this does not work I have got no other choice"'